In [1]:
import pathlib

In [2]:
import numpy as np
import pandas as pd

/tmp/nix-shell.PYG4dJ/ipykernel_907647/1662815981.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
data_folder = pathlib.Path("../data/processed/biolog/")
ch_pheno_file = data_folder / "phenotypes/ch_phenotypes.tsv"
ch_unitrembl_file = data_folder / "features/uniprot_trembl/uniprot_trembl_ch_features.tsv"
ch_pheno_file.is_file(), ch_unitrembl_file.is_file()

(True, True)

In [4]:
from trait_prediction.main import PhenotypeSet

In [5]:
# Load the phenotype data
phenotypeset = PhenotypeSet.read_data(ch_pheno_file)

In [6]:
phenotype_sizes = []
for phenotype in phenotypeset.phenotypes:
    phenotype_sizes.append(phenotype.phenotype_data.size)
min(phenotype_sizes), max(phenotype_sizes)

(103, 354)

In [7]:
phenotype1 = list(phenotypeset.phenotypes)[0]
phenotype1

Phenotype (name=Carbon-D-Trehalose, category=ch_biolog, size=323)

In [8]:
from trait_prediction.utils import read_generic_features
from trait_prediction.feature_selection import remove_features_with_low_variance, remove_features_with_high_correlation, feature_selection_kbest

In [9]:
VARIANCE_THRESHOLD = 0.01
CORRELATION_THRESHOLD = 0.95
TEST_SIZE = 0.3
N_SPLITS = 5
PHENOTYPE_SAMPLE_SIZE_THRESHOLD = 10
MINOR_CLASS_SAMPLE_SIZE_THRESHOLD = 5
SHAP_MAX_DISPLAY = 10

In [26]:
raw_features = read_generic_features(ch_unitrembl_file, bool_conversion=True, dtype="uint8")
raw_features.shape

(341, 155177)

In [27]:
raw_features.memory_usage(deep=True).sum() / (1024**2)

50.493215560913086

In [11]:
raw_features, _ = remove_features_with_low_variance(raw_features, VARIANCE_THRESHOLD)
raw_features.shape

(341, 26014)

In [12]:
%%time
features = read_generic_features(ch_unitrembl_file)

CPU times: user 34.3 s, sys: 1e+03 ms, total: 35.3 s
Wall time: 35.3 s


In [13]:
%%time
features, low_var_features1 = remove_features_with_low_variance(features, threshold=VARIANCE_THRESHOLD)

CPU times: user 464 ms, sys: 50 ms, total: 513 ms
Wall time: 513 ms


In [14]:
len(low_var_features1)

129163

In [15]:
%%time
features, correlated_features_dict1 = remove_features_with_high_correlation(features, threshold=CORRELATION_THRESHOLD)

CPU times: user 3min 56s, sys: 2.1 s, total: 3min 58s
Wall time: 3min 58s


In [16]:
features.shape

(341, 3322)

In [17]:
phenotype1.set_feature_data(features, feature_type="generic")

In [18]:
%%time
(
    low_var_features2,
    correlated_features_dict2,
    low_score_features,
) = phenotype1.filter_feature_data(
    variance_threshold=VARIANCE_THRESHOLD,
    correlation_treshold=CORRELATION_THRESHOLD,
    score_func="chi2",
    n_features=1000,
)

CPU times: user 3.31 s, sys: 260 ms, total: 3.57 s
Wall time: 3.27 s


In [19]:
%%time
low_var_features = set(low_var_features1 + low_var_features2)
correlated_features_dict = {
    **correlated_features_dict1,
    **correlated_features_dict2,
}

CPU times: user 21.1 ms, sys: 73.1 ms, total: 94.2 ms
Wall time: 4.23 ms


## Benchmarking correlation function

In [20]:
%%time
corr_matrix = raw_features.corr().abs()

CPU times: user 3min 51s, sys: 1.02 s, total: 3min 52s
Wall time: 3min 51s


In [21]:
corr_matrix

,tr|A0A9E8GM07|A0A9E8GM07_9PSED,tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,tr|A0A2N8GX02|A0A2N8GX02_9PSED,tr|A0A9E8GT22|A0A9E8GT22_9PSED,tr|A0A9E8K6N1|A0A9E8K6N1_9PSED,tr|A0A9E8GRF5|A0A9E8GRF5_9PSED,tr|A0A9E8BB96|A0A9E8BB96_9PSED,tr|A0A9E8GRQ8|A0A9E8GRQ8_9PSED,tr|A0A9E8BK21|A0A9E8BK21_9PSED,...,tr|A0A0H3NRM4|A0A0H3NRM4_YERE1,tr|F4MUQ6|F4MUQ6_YEREN,tr|A0A8B6L0C5|A0A8B6L0C5_YEREN,tr|F4N0G1|F4N0G1_YEREN,tr|A0A0H5G3Z2|A0A0H5G3Z2_YEREN,tr|F4MWV5|F4MWV5_YEREN,tr|F4N7V8|F4N7V8_YEREN,tr|F4MXP5|F4MXP5_YEREN,tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,tr|A0A0E8LY24|A0A0E8LY24_YEREN
tr|A0A9E8GM07|A0A9E8GM07_9PSED,1.000000,0.797053,0.869126,0.758812,0.676661,0.718787,0.970324,0.584253,0.968649,0.475623,...,0.033161,0.039814,0.030655,0.027943,0.027943,0.035504,0.033161,0.033161,0.024956,0.043747
tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,0.797053,1.000000,0.830400,0.755270,0.519984,0.694724,0.773400,0.606775,0.744363,0.596727,...,0.026431,0.031734,0.024434,0.022272,0.022272,0.028298,0.026431,0.026431,0.019891,0.034868
tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,0.869126,0.830400,1.000000,0.782307,0.677377,0.827023,0.843334,0.555752,0.824839,0.547243,...,0.028821,0.034604,0.026643,0.024286,0.024286,0.030857,0.028821,0.028821,0.021690,0.038021
tr|A0A2N8GX02|A0A2N8GX02_9PSED,0.758812,0.755270,0.782307,1.000000,0.662068,0.621963,0.736294,0.637755,0.701178,0.465366,...,0.025163,0.030211,0.023262,0.021203,0.021203,0.026941,0.025163,0.025163,0.018937,0.033195
tr|A0A9E8GT22|A0A9E8GT22_9PSED,0.676661,0.519984,0.677377,0.662068,1.000000,0.457936,0.656580,0.716073,0.698561,0.343006,...,0.022439,0.026941,0.020743,0.018908,0.018908,0.024024,0.022439,0.022439,0.016886,0.029602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tr|F4MWV5|F4MWV5_YEREN,0.035504,0.028298,0.030857,0.026941,0.024024,0.025520,0.036590,0.020743,0.034391,0.016886,...,0.934013,0.891737,0.421347,0.787032,0.787032,1.000000,0.934013,0.934013,0.702898,0.811578
tr|F4N7V8|F4N7V8_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|F4MXP5|F4MXP5_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,0.024956,0.019891,0.021690,0.018937,0.016886,0.017938,0.025719,0.014580,0.024173,0.011869,...,0.752557,0.626800,0.399745,0.666502,0.666502,0.702898,0.752557,0.752557,1.000000,0.570456


In [22]:
del corr_matrix

In [23]:
def pearson_correlation_coefficient(X):
    """
    calculate pearson correlation coefficient of matrix X
    X: numpy array (MxN)
    return pcc: numpy array (MxM)
    """
    M, N = X.shape[0], X.shape[1]  # number of features, number of data points
    X_mean = np.mean(X, axis=1).reshape(M, 1)
    X_std = np.std(X, axis=1).reshape(M, 1)
    X_tilde = (X-X_mean)/X_std
    pcc = X_tilde@X_tilde.T/N
    np.fill_diagonal(pcc, 1, wrap=False)
    return pcc

In [24]:
%%time
corr_matrix = pd.DataFrame(np.abs(pearson_correlation_coefficient(raw_features.values.T)), index=raw_features.columns, columns=raw_features.columns)

CPU times: user 7.42 s, sys: 7.46 s, total: 14.9 s
Wall time: 1.9 s


In [25]:
corr_matrix

,tr|A0A9E8GM07|A0A9E8GM07_9PSED,tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,tr|A0A2N8GX02|A0A2N8GX02_9PSED,tr|A0A9E8GT22|A0A9E8GT22_9PSED,tr|A0A9E8K6N1|A0A9E8K6N1_9PSED,tr|A0A9E8GRF5|A0A9E8GRF5_9PSED,tr|A0A9E8BB96|A0A9E8BB96_9PSED,tr|A0A9E8GRQ8|A0A9E8GRQ8_9PSED,tr|A0A9E8BK21|A0A9E8BK21_9PSED,...,tr|A0A0H3NRM4|A0A0H3NRM4_YERE1,tr|F4MUQ6|F4MUQ6_YEREN,tr|A0A8B6L0C5|A0A8B6L0C5_YEREN,tr|F4N0G1|F4N0G1_YEREN,tr|A0A0H5G3Z2|A0A0H5G3Z2_YEREN,tr|F4MWV5|F4MWV5_YEREN,tr|F4N7V8|F4N7V8_YEREN,tr|F4MXP5|F4MXP5_YEREN,tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,tr|A0A0E8LY24|A0A0E8LY24_YEREN
tr|A0A9E8GM07|A0A9E8GM07_9PSED,1.000000,0.797053,0.869126,0.758812,0.676661,0.718787,0.970324,0.584253,0.968649,0.475623,...,0.033161,0.039814,0.030655,0.027943,0.027943,0.035504,0.033161,0.033161,0.024956,0.043747
tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,0.797053,1.000000,0.830400,0.755270,0.519984,0.694724,0.773400,0.606775,0.744363,0.596727,...,0.026431,0.031734,0.024434,0.022272,0.022272,0.028298,0.026431,0.026431,0.019891,0.034868
tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,0.869126,0.830400,1.000000,0.782307,0.677377,0.827023,0.843334,0.555752,0.824839,0.547243,...,0.028821,0.034604,0.026643,0.024286,0.024286,0.030857,0.028821,0.028821,0.021690,0.038021
tr|A0A2N8GX02|A0A2N8GX02_9PSED,0.758812,0.755270,0.782307,1.000000,0.662068,0.621963,0.736294,0.637755,0.701178,0.465366,...,0.025163,0.030211,0.023262,0.021203,0.021203,0.026941,0.025163,0.025163,0.018937,0.033195
tr|A0A9E8GT22|A0A9E8GT22_9PSED,0.676661,0.519984,0.677377,0.662068,1.000000,0.457936,0.656580,0.716073,0.698561,0.343006,...,0.022439,0.026941,0.020743,0.018908,0.018908,0.024024,0.022439,0.022439,0.016886,0.029602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tr|F4MWV5|F4MWV5_YEREN,0.035504,0.028298,0.030857,0.026941,0.024024,0.025520,0.036590,0.020743,0.034391,0.016886,...,0.934013,0.891737,0.421347,0.787032,0.787032,1.000000,0.934013,0.934013,0.702898,0.811578
tr|F4N7V8|F4N7V8_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|F4MXP5|F4MXP5_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,0.024956,0.019891,0.021690,0.018937,0.016886,0.017938,0.025719,0.014580,0.024173,0.011869,...,0.752557,0.626800,0.399745,0.666502,0.666502,0.702898,0.752557,0.752557,1.000000,0.570456


In [26]:
%%time
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_group_dict = {}
cols_to_drop_set = set()
for col in upper.columns:
    corr_filter = upper[col] > CORRELATION_THRESHOLD
    correlated_cols = list(upper.columns[corr_filter])
    if len(correlated_cols) > 0:
        corr_group_dict[col] = correlated_cols
        cols_to_drop_set.add(col)

CPU times: user 5.24 s, sys: 1.16 s, total: 6.4 s
Wall time: 6.4 s


## Making correlation calculation super fast

In [27]:
from numba import jit

In [28]:
def pearson_correlation_coefficient(X):
    """
    calculate pearson correlation coefficient of matrix X
    X: numpy array (MxN)
    return pcc: numpy array (MxM)
    """
    M, N = X.shape[0], X.shape[1]  # number of features, number of data points
    X_mean = np.mean(X, axis=1).reshape(M, 1)
    X_std = np.std(X, axis=1).reshape(M, 1)
    X_tilde = (X-X_mean)/X_std
    pcc = X_tilde@X_tilde.T/N
    np.fill_diagonal(pcc, 1, wrap=False)
    return pcc

In [41]:
import numba
from numba.typed import Dict

@jit(nopython=True)
def find_columns_to_drop(corr_matrix, threshold):
    # Initialize an empty set to hold indices of columns to drop
    cols_to_drop = set()
    corr_groups = []
    # Get the number of columns in the correlation matrix
    n_cols = corr_matrix.shape[1]
    # Iterate over columns
    for i in range(n_cols):
        corr_set = set()
        for j in range(i + 1, n_cols):
            # If correlation is above the threshold and not already in the list
            if corr_matrix[i, j] >= threshold:
                corr_set.add(j)
                if j not in cols_to_drop:
                    cols_to_drop.add(j)
        corr_groups.append(corr_set)
    return list(cols_to_drop), corr_groups

In [72]:
def create_corr_group_dict(corr_groups, cols):
    """Convert list of sets to dict of cols"""
    corr_group_dict = {}
    for i, corr_group in enumerate(corr_groups):
        if len(corr_group) < 1:
            continue
        corr_group_dict[cols[i]] = list(cols[list(corr_group)])
    return corr_group_dict

In [73]:
def remove_correlated_columns_numba(df, threshold=0.95):
    # Calculate the correlation matrix and convert it to a NumPy array
    corr_matrix = np.abs(pearson_correlation_coefficient(df.to_numpy().T))
    # Find indices of columns to drop
    to_drop, corr_groups = find_columns_to_drop(corr_matrix, threshold)
    corr_group_dict = create_corr_group_dict(corr_groups, df.columns)
    # Drop the columns from the DataFrame
    df_reduced = df.drop(df.columns[to_drop], axis=1)
    return df_reduced, corr_group_dict

In [74]:
%%prun
df_reduced, corr_group_dict = remove_correlated_columns_numba(raw_features, threshold=0.95)
df_reduced.shape

         663259 function calls (663248 primitive calls) in 2.950 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    1.437    1.437    1.464    1.464 2120553404.py:1(pearson_correlation_coefficient)
        1    0.555    0.555    2.895    2.895 1605592884.py:1(remove_correlated_columns_numba)
        1    0.429    0.429    0.430    0.430 311124116.py:4(find_columns_to_drop)
        1    0.181    0.181    0.443    0.443 4228291155.py:1(create_corr_group_dict)
    45387    0.141    0.000    0.210    0.000 base.py:5369(__getitem__)
        1    0.055    0.055    2.950    2.950 <string>:1(<module>)
    22693    0.042    0.000    0.047    0.000 base.py:836(__iter__)
    22695    0.016    0.000    0.022    0.000 base.py:649(_simple_new)
        1    0.014    0.014    0.021    0.021 _methods.py:198(_var)
   113661    0.014    0.000    0.035    0.000 {built-in method builtins.isinstance}
        5    0.012    0.002    0.01

In [46]:
df_reduced

,tr|A0A9E8GM07|A0A9E8GM07_9PSED,tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,tr|A0A2N8GX02|A0A2N8GX02_9PSED,tr|A0A9E8GT22|A0A9E8GT22_9PSED,tr|A0A9E8K6N1|A0A9E8K6N1_9PSED,tr|A0A9E8BB96|A0A9E8BB96_9PSED,tr|A0A9E8BK21|A0A9E8BK21_9PSED,tr|A0A9E8B7A3|A0A9E8B7A3_9PSED,tr|A0A2N8GW12|A0A2N8GW12_9PSED,...,tr|A0A0E8KEX0|A0A0E8KEX0_YEREN,tr|F4N4B0|F4N4B0_YEREN,tr|A0A0H3B501|A0A0H3B501_YERPY,tr|F4N475|F4N475_YEREN,tr|F4N589|F4N589_YEREN,tr|F4N2X7|F4N2X7_YEREN,tr|A0A7U0AUA7|A0A7U0AUA7_YEREN,tr|F4MVJ6|F4MVJ6_YEREN,tr|F4N738|F4N738_YEREN,tr|A0A8B6L0C5|A0A8B6L0C5_YEREN
genomeID,,,,,,,,,,,,,,,,,,,,,
149698.22,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
702115.9,1,1,1,1,1,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1663.218,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
242605.18,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
134536.40,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1663.234,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
134536.48,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
754262.3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [49]:
raw_features.shape

(341, 26014)